## Feature Engineering

This notebook creates lagged, rolling, trend, usage, durability, age, and team-context features for predicting future full-PPR fantasy points. All features for season t must use information available before season t.

In [128]:
import polars as pl

df = pl.read_csv("../data/processed/player_stats_clean_2016_2025.csv")

In [129]:
# Core Lagged Features

df = df.sort(["player_id", "season"])

df = df.with_columns([
    pl.col("fantasy_points_ppr_calc")
      .shift(1)
      .over("player_id")
      .alias("fantasy_points_lag_1"),

    pl.col("fantasy_points_ppr_calc")
      .shift(2)
      .over("player_id")
      .alias("fantasy_points_lag_2"),

    pl.col("games")
      .shift(1)
      .over("player_id")
      .alias("games_lag_1")
])

In [130]:
# Prior year PPG

df = df.with_columns(
    (
        pl.col("fantasy_points_ppr_calc")
        / pl.col("games")
    ).alias("fantasy_ppg")
)

df = df.with_columns(
    pl.col("fantasy_ppg")
      .shift(1)
      .over("player_id")
      .alias("fantasy_ppg_lag_1")
)

In [131]:
# Weighted two year feature

df = df.with_columns(
    (
        0.7 * pl.col("fantasy_points_lag_1")
        + 0.3 * pl.col("fantasy_points_lag_2")
    ).alias("fantasy_points_2yr_weighted")
)

In [132]:
df.select([
    "player_display_name",
    "season",
    "fantasy_points_ppr_calc",
    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "games_lag_1",
    "fantasy_points_2yr_weighted"
]).head(20)

player_display_name,season,fantasy_points_ppr_calc,fantasy_points_lag_1,fantasy_points_lag_2,fantasy_ppg_lag_1,games_lag_1,fantasy_points_2yr_weighted
str,i64,f64,f64,f64,f64,i64,f64
"""Tom Brady""",2016,258.56,null,null,null,null,null
"""Tom Brady""",2017,295.88,258.56,null,21.546667,12,null
"""Tom Brady""",2018,281.3,295.88,258.56,18.4925,16,284.684
"""Tom Brady""",2019,263.68,281.3,295.88,17.58125,16,285.674
"""Tom Brady""",2020,337.92,263.68,281.3,16.48,16,268.966
…,…,…,…,…,…,…,…
"""Josh McCown""",2017,205.44,52.1,null,10.42,5,null
"""Josh McCown""",2018,20.76,205.44,52.1,15.803077,13,159.438
"""Josh McCown""",2019,0.76,20.76,205.44,5.19,4,76.164


# Volume/Opportunity Features

In [133]:
df = df.with_columns([
    pl.col("targets")
      .shift(1)
      .over("player_id")
      .alias("targets_lag_1"),

    pl.col("carries")
      .shift(1)
      .over("player_id")
      .alias("carries_lag_1"),

    pl.col("receptions")
      .shift(1)
      .over("player_id")
      .alias("receptions_lag_1")
])

In [134]:
# Combined Opportunity Feature

df = df.with_columns(
    (
        pl.col("targets_lag_1")
        + pl.col("carries_lag_1")
    ).alias("opportunities_lag_1")
)

In [135]:
# Per game versions

df = df.with_columns([
    (
        pl.col("targets_lag_1")
        / pl.col("games_lag_1")
    ).alias("targets_per_game_lag_1"),

    (
        pl.col("carries_lag_1")
        / pl.col("games_lag_1")
    ).alias("carries_per_game_lag_1"),

    (
        pl.col("opportunities_lag_1")
        / pl.col("games_lag_1")
    ).alias("opportunities_per_game_lag_1")
])

In [136]:
df.select([
    "player_display_name",
    "season",
    "position",
    "targets_lag_1",
    "carries_lag_1",
    "opportunities_lag_1",
    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1"
]).head(20)

player_display_name,season,position,targets_lag_1,carries_lag_1,opportunities_lag_1,targets_per_game_lag_1,carries_per_game_lag_1,opportunities_per_game_lag_1
str,i64,str,i64,i64,i64,f64,f64,f64
"""Tom Brady""",2016,"""QB""",null,null,null,null,null,null
"""Tom Brady""",2017,"""QB""",0,28,28,0.0,2.333333,2.333333
"""Tom Brady""",2018,"""QB""",0,25,25,0.0,1.5625,1.5625
"""Tom Brady""",2019,"""QB""",1,23,24,0.0625,1.4375,1.5
"""Tom Brady""",2020,"""QB""",0,26,26,0.0,1.625,1.625
…,…,…,…,…,…,…,…,…
"""Josh McCown""",2017,"""QB""",0,7,7,0.0,1.4,1.4
"""Josh McCown""",2018,"""QB""",0,37,37,0.0,2.846154,2.846154
"""Josh McCown""",2019,"""QB""",0,5,5,0.0,1.25,1.25


In [137]:
df = df.with_columns([
    pl.col("target_share")
      .shift(1)
      .over("player_id")
      .alias("target_share_lag_1"),

    pl.col("air_yards_share")
      .shift(1)
      .over("player_id")
      .alias("air_yards_share_lag_1"),

    pl.col("wopr")
      .shift(1)
      .over("player_id")
      .alias("wopr_lag_1")
])

In [138]:
df = df.with_columns([
    (
        pl.col("targets_lag_1")
        - pl.col("targets")
          .shift(2)
          .over("player_id")
    ).alias("targets_change"),

    (
        pl.col("fantasy_points_lag_1")
        - pl.col("fantasy_points_lag_2")
    ).alias("fantasy_points_change")
])

In [139]:
df.select([
    "player_display_name",
    "season",
    "position",
    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",
    "targets_change",
    "fantasy_points_change"
]).head(20)

player_display_name,season,position,target_share_lag_1,air_yards_share_lag_1,wopr_lag_1,targets_change,fantasy_points_change
str,i64,str,f64,f64,f64,i64,f64
"""Tom Brady""",2016,"""QB""",null,null,null,null,null
"""Tom Brady""",2017,"""QB""",0.0,0.0,0.0,null,null
"""Tom Brady""",2018,"""QB""",0.0,0.0,0.0,0,37.32
"""Tom Brady""",2019,"""QB""",0.001799,0.0,0.002698,1,-14.58
"""Tom Brady""",2020,"""QB""",0.0,0.0,0.0,-1,-17.62
…,…,…,…,…,…,…,…
"""Josh McCown""",2017,"""QB""",0.0,0.0,0.0,null,null
"""Josh McCown""",2018,"""QB""",0.0,0.0,0.0,0,153.34
"""Josh McCown""",2019,"""QB""",0.0,0.0,0.0,0,-184.68


# Age, Experience, and Team Change Features

In [140]:
import nflreadpy as nfl

seasons = list(range(2016, 2026))
rosters = nfl.load_rosters(seasons=seasons)

In [141]:
roster_season = (
    rosters
    .select([
        "season",
        "gsis_id",
        "birth_date",
        "years_exp",
        "entry_year",
        "rookie_year",
        "draft_number"
    ])
    .unique(
        subset=["season", "gsis_id"],
        keep="first"
    )
)

In [142]:
df = df.join(
    roster_season,
    left_on=["player_id", "season"],
    right_on=["gsis_id", "season"],
    how="left"
)

In [143]:
df = df.with_columns(
    (
        (
            pl.date(pl.col("season"), 9, 1)
            - pl.col("birth_date")
        )
        .dt.total_days()
        / 365.25
    ).alias("age")
)

In [144]:
df = df.with_columns(
    (pl.col("age") ** 2).alias("age_squared")
)

In [145]:
df = df.sort(["player_id", "season"])

df = df.with_columns(
    pl.col("recent_team")
      .shift(1)
      .over("player_id")
      .alias("previous_team")
)

df = df.with_columns(
    (
        pl.col("previous_team").is_not_null()
        & (pl.col("recent_team") != pl.col("previous_team"))
    )
    .cast(pl.Int8)
    .alias("team_change_flag")
)

In [146]:
df.select([
    "player_display_name",
    "season",
    "position",
    "age",
    "age_squared",
    "years_exp",
    "draft_number",
    "recent_team",
    "previous_team",
    "team_change_flag"
]).head(20)

player_display_name,season,position,age,age_squared,years_exp,draft_number,recent_team,previous_team,team_change_flag
str,i64,str,f64,f64,i32,i32,str,str,i8
"""Tom Brady""",2016,"""QB""",39.080082,1527.25282,16,null,"""NE""",null,0
"""Tom Brady""",2017,"""QB""",40.079398,1606.358118,17,199,"""NE""","""NE""",0
"""Tom Brady""",2018,"""QB""",41.078713,1687.460679,18,199,"""NE""","""NE""",0
"""Tom Brady""",2019,"""QB""",42.078029,1770.560503,19,199,"""NE""","""NE""",0
"""Tom Brady""",2020,"""QB""",43.080082,1855.893477,20,199,"""TB""","""NE""",1
…,…,…,…,…,…,…,…,…,…
"""Josh McCown""",2017,"""QB""",38.162902,1456.407098,15,null,"""NYJ""","""CLE""",1
"""Josh McCown""",2018,"""QB""",39.162218,1533.679292,16,81,"""NYJ""","""NYJ""",0
"""Josh McCown""",2019,"""QB""",40.161533,1612.948749,17,81,"""PHI""","""NYJ""",1


# Efficiency Features

In [147]:
df = df.with_columns([
    pl.when(pl.col("targets") > 0)
      .then(pl.col("receiving_yards") / pl.col("targets"))
      .otherwise(None)
      .alias("yards_per_target"),

    pl.when(pl.col("carries") > 0)
      .then(pl.col("rushing_yards") / pl.col("carries"))
      .otherwise(None)
      .alias("yards_per_carry"),

    pl.when(pl.col("targets") > 0)
      .then(pl.col("receptions") / pl.col("targets"))
      .otherwise(None)
      .alias("catch_rate"),

    pl.when((pl.col("targets") + pl.col("carries")) > 0)
      .then(
          pl.col("fantasy_points_ppr_calc")
          / (pl.col("targets") + pl.col("carries"))
      )
      .otherwise(None)
      .alias("fantasy_points_per_opportunity")
])

In [148]:
df = df.sort(["player_id", "season"])

df = df.with_columns([
    pl.col("yards_per_target")
      .shift(1)
      .over("player_id")
      .alias("yards_per_target_lag_1"),

    pl.col("yards_per_carry")
      .shift(1)
      .over("player_id")
      .alias("yards_per_carry_lag_1"),

    pl.col("catch_rate")
      .shift(1)
      .over("player_id")
      .alias("catch_rate_lag_1"),

    pl.col("fantasy_points_per_opportunity")
      .shift(1)
      .over("player_id")
      .alias("fantasy_points_per_opportunity_lag_1")
])

In [149]:
df.select([
    "player_display_name",
    "season",
    "position",
    "yards_per_target_lag_1",
    "yards_per_carry_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1"
]).head(20)

player_display_name,season,position,yards_per_target_lag_1,yards_per_carry_lag_1,catch_rate_lag_1,fantasy_points_per_opportunity_lag_1
str,i64,str,f64,f64,f64,f64
"""Tom Brady""",2016,"""QB""",null,null,null,null
"""Tom Brady""",2017,"""QB""",null,2.285714,null,9.234286
"""Tom Brady""",2018,"""QB""",null,1.12,null,11.8352
"""Tom Brady""",2019,"""QB""",6.0,1.521739,1.0,11.720833
"""Tom Brady""",2020,"""QB""",null,1.307692,null,10.141538
…,…,…,…,…,…,…
"""Josh McCown""",2017,"""QB""",null,3.0,null,7.442857
"""Josh McCown""",2018,"""QB""",null,3.351351,null,5.552432
"""Josh McCown""",2019,"""QB""",null,6.4,null,4.152


# Historical Rolling Features

In [150]:
df = df.with_columns([
    (
        (
            pl.col("games").shift(1).over("player_id")
            + pl.col("games").shift(2).over("player_id")
        ) / 2
    ).alias("games_2yr_avg"),

    (
        (
            pl.col("targets").shift(1).over("player_id")
            + pl.col("targets").shift(2).over("player_id")
        ) / 2
    ).alias("targets_2yr_avg"),

    (
        (
            pl.col("carries").shift(1).over("player_id")
            + pl.col("carries").shift(2).over("player_id")
        ) / 2
    ).alias("carries_2yr_avg")
])

In [151]:
df = df.with_columns(
    (
        pl.col("targets_2yr_avg")
        + pl.col("carries_2yr_avg")
    ).alias("opportunities_2yr_avg")
)

In [152]:
df = df.with_columns([
    (
        (
            pl.col("target_share").shift(1).over("player_id")
            + pl.col("target_share").shift(2).over("player_id")
        ) / 2
    ).alias("target_share_2yr_avg"),

    (
        (
            pl.col("wopr").shift(1).over("player_id")
            + pl.col("wopr").shift(2).over("player_id")
        ) / 2
    ).alias("wopr_2yr_avg")
])

In [153]:
df.select([
    "player_display_name",
    "season",
    "position",
    "games_2yr_avg",
    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg",
    "target_share_2yr_avg",
    "wopr_2yr_avg"
]).head(20)

player_display_name,season,position,games_2yr_avg,targets_2yr_avg,carries_2yr_avg,opportunities_2yr_avg,target_share_2yr_avg,wopr_2yr_avg
str,i64,str,f64,f64,f64,f64,f64,f64
"""Tom Brady""",2016,"""QB""",null,null,null,null,null,null
"""Tom Brady""",2017,"""QB""",null,null,null,null,null,null
"""Tom Brady""",2018,"""QB""",14.0,0.0,26.5,26.5,0.0,0.0
"""Tom Brady""",2019,"""QB""",16.0,0.5,24.0,24.5,0.000899,0.001349
"""Tom Brady""",2020,"""QB""",16.0,0.5,24.5,25.0,0.000899,0.001349
…,…,…,…,…,…,…,…,…
"""Josh McCown""",2017,"""QB""",null,null,null,null,null,null
"""Josh McCown""",2018,"""QB""",9.0,0.0,22.0,22.0,0.0,0.0
"""Josh McCown""",2019,"""QB""",8.5,0.0,21.0,21.0,0.0,0.0


# Candidate Feature Set

In [154]:
candidate_features = [
    # Player context
    "age",
    "age_squared",
    "years_exp",
    "draft_number",
    "team_change_flag",

    # Prior production
    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    # Durability
    "games_lag_1",
    "games_2yr_avg",

    # Opportunity
    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",
    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",

    # Advanced receiving usage
    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",
    "target_share_2yr_avg",
    "wopr_2yr_avg",

    # Trends
    "targets_change",
    "fantasy_points_change",

    # Efficiency
    "yards_per_target_lag_1",
    "yards_per_carry_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    # Rolling opportunity
    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg",
]

In [155]:
# Target variable

target = "fantasy_points_ppr_calc"

In [156]:
# Model ready subset (version 1 will require 2 seasons played)

model_df = df.filter(
    pl.col("fantasy_points_lag_2").is_not_null()
)

In [157]:
print("Full dataset:", df.shape)
print("Model dataset:", model_df.shape)

model_df.group_by("position").len().sort("position")

Full dataset: (5874, 74)
Model dataset: (2909, 74)


position,len
str,u32
"""QB""",430
"""RB""",726
"""TE""",635
"""WR""",1118


In [158]:
model_df.select(candidate_features).null_count()

age,age_squared,years_exp,draft_number,team_change_flag,fantasy_points_lag_1,fantasy_points_lag_2,fantasy_ppg_lag_1,fantasy_points_2yr_weighted,games_lag_1,games_2yr_avg,targets_lag_1,carries_lag_1,receptions_lag_1,opportunities_lag_1,targets_per_game_lag_1,carries_per_game_lag_1,opportunities_per_game_lag_1,target_share_lag_1,air_yards_share_lag_1,wopr_lag_1,target_share_2yr_avg,wopr_2yr_avg,targets_change,fantasy_points_change,yards_per_target_lag_1,yards_per_carry_lag_1,catch_rate_lag_1,fantasy_points_per_opportunity_lag_1,targets_2yr_avg,carries_2yr_avg,opportunities_2yr_avg
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,814,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,469,1152,469,79,0,0,0


# Position Specific Feature Lists

In [159]:
base_features = [
    "age",
    "age_squared",
    "years_exp",
    "draft_number",
    "team_change_flag",

    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    "games_lag_1",
    "games_2yr_avg",

    "fantasy_points_change"
]

In [160]:
# QB

qb_features = base_features + [
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg"
]

In [161]:
# RB

rb_features = base_features + [
    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",

    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",

    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

In [162]:
# WR

wr_features = base_features + [
    "targets_lag_1",
    "receptions_lag_1",
    "targets_per_game_lag_1",

    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",

    "targets_change",

    "yards_per_target_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "target_share_2yr_avg",
    "wopr_2yr_avg"
]

In [163]:
# TE

te_features = wr_features.copy()

In [164]:
# Undrafted Players

df = df.with_columns(
    pl.col("draft_number")
      .is_null()
      .cast(pl.Int8)
      .alias("undrafted_flag")
)

In [165]:
df = df.with_columns(
    pl.col("draft_number")
      .fill_null(260)
      .alias("draft_number_filled")
)

In [166]:
base_features = [
    "age",
    "age_squared",
    "years_exp",

    "draft_number_filled",
    "undrafted_flag",

    "team_change_flag",

    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    "games_lag_1",
    "games_2yr_avg",

    "fantasy_points_change"
]

In [167]:
# QB

qb_features = base_features + [
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg"
]

In [168]:
# RB

rb_features = base_features + [
    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",

    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",

    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

In [169]:
# WR

wr_features = base_features + [
    "targets_lag_1",
    "receptions_lag_1",
    "targets_per_game_lag_1",

    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",

    "targets_change",

    "yards_per_target_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "target_share_2yr_avg",
    "wopr_2yr_avg"
]

In [170]:
# TE

te_features = wr_features.copy()

In [171]:
model_df = df.filter(
    pl.col("fantasy_points_lag_2").is_not_null()
)

In [172]:
position_feature_sets = {
    "QB": qb_features,
    "RB": rb_features,
    "WR": wr_features,
    "TE": te_features
}

for position, features in position_feature_sets.items():

    pos_df = model_df.filter(
        pl.col("position") == position
    )

    print(f"\n{position}")
    print(
        pos_df
        .select(features)
        .null_count()
    )


QB
shape: (1, 16)
┌─────┬────────────┬───────────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ age ┆ age_square ┆ years_exp ┆ draft_numb ┆ … ┆ fantasy_po ┆ carries_la ┆ carries_pe ┆ carries_2 │
│ --- ┆ d          ┆ ---       ┆ er_filled  ┆   ┆ ints_chang ┆ g_1        ┆ r_game_lag ┆ yr_avg    │
│ u32 ┆ ---        ┆ u32       ┆ ---        ┆   ┆ e          ┆ ---        ┆ _1         ┆ ---       │
│     ┆ u32        ┆           ┆ u32        ┆   ┆ ---        ┆ u32        ┆ ---        ┆ u32       │
│     ┆            ┆           ┆            ┆   ┆ u32        ┆            ┆ u32        ┆           │
╞═════╪════════════╪═══════════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 0   ┆ 0          ┆ 0         ┆ 0          ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0         │
└─────┴────────────┴───────────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘

RB
shape: (1, 25)
┌─────┬────────────┬───────────┬────────────┬───┬────

# Saving Full Dataset

In [173]:
model_path = "../data/processed/modeling_dataset_2018_2025.csv"

model_df.write_csv(model_path)

In [174]:
import os
print(os.path.exists(model_path))
print(model_df.shape)

True
(2909, 76)
